In [1]:
import requests
import json
import pandas as pd
import os
import tqdm as tqdm
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import quote_plus


# Define your API key
api_key = 'ntioungsta'

In [2]:
# Load dataset
df = pd.read_csv('/home/sbasir/Thesis/Thesis/EDP/europeana_datasets - europeana_datasets.csv')
df.head()

,id,total_count
0,2058621_Ag_EU_LoCloud_NRA,2910176
1,9200365_Ag_EU_TEL_a0142_Gallica,1191745
2,9200359_Ag_EU_TEL_a0601_Newspapers_Netherlands,747773
3,9200479_NLPoland,644682
4,9200384_Ag_EU_TEL_a0613_Newspapers_ONB,629498


In [3]:
# # get last 500 records
# df_firsts = df[:1292]
# total_count = df_firsts['total_count'].sum()
# print('Total:', total_count)

# df1 = df_firsts[:10]
# df2 = df_firsts[10:50]
# df3 = df_firsts[50:100]
# df4 = df_firsts[100:200]
# df5 = df_firsts[200:400]
# df6 = df_firsts[400:600]
# df7 = df_firsts[600:800]
# df8 = df_firsts[800:1000]
# df9 = df_firsts[1000:1292]
# df10 = df[1292:]

total_datasets = len(df)
print(total_datasets)

df1 = df[:718]
df2 = df[718:1436]
df3 = df[1436:]

2154


In [5]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
from urllib.parse import quote_plus

df = df1

N = 1000000

num_datasets = len(df)
documents_per_dataset = max(1, N // num_datasets)

rows = 100

document_ids = []
documents = []

max_requests = 100  

max_worker = 100


print(f"Number of documents to fetch: {N}")
print(f"Number of documents per dataset: {documents_per_dataset}")
print(f"Number of datasets: {num_datasets}")
print(f"Number of requests per dataset: {max_requests}")

# Function to get documents using cursor-based pagination
def get_documents(api_key, dataset_name, rows, cursor):
    start_time = time.time()
    search_url = f'https://api.europeana.eu/record/v2/search.json?wskey={api_key}&query=*&qf=edm_datasetName:"{dataset_name}"&rows={rows}&cursor={cursor}&profile=minimal&sort=random_1 asc, europeana_id asc'
    response = requests.get(search_url)
    search_time = time.time() - start_time
    
    if response.status_code == 429:  
        print("Rate limit exceeded. Waiting before retrying...")
        time.sleep(60)  
        return [], cursor, search_time

    data = response.json()
    if 'items' in data:
        document_ids = [item['id'] for item in data['items']]
    else:
        document_ids = []
    
    next_cursor = data.get('nextCursor', None)
    safe_next_cursor = quote_plus(next_cursor) if next_cursor else None
    return document_ids, safe_next_cursor, search_time

def fetch_document(api_key, doc_id):
    start_time = time.time()
    record_url = f'https://api.europeana.eu/record/v2/{doc_id}.rdf?wskey={api_key}'
    response = requests.get(record_url)
    record_time = time.time() - start_time
    
    if response.status_code == 429:  
        print(f"Rate limit exceeded while fetching document {doc_id}. Waiting before retrying...")
        time.sleep(60)  
        return None, record_time

    return response.text, record_time

total_search_time = 0
total_record_time = 0

for index, row in df.iterrows():
    dataset_name = row[0]
    cursor = '*'
    current_request = 0
    dataset_documents = []

    print(f'Collecting from dataset: {dataset_name}')
    
    while (cursor and current_request < max_requests):
        print('number of documents from this dataset:', len(dataset_documents))
        print(f'Request #{current_request + 1} for dataset {dataset_name}')
        current_request += 1
        new_document_ids, cursor, search_time = get_documents(api_key, dataset_name, rows, cursor)
        total_search_time += search_time
        
        document_ids.extend(new_document_ids)
        print('total number of documents:', len(document_ids))

        with ThreadPoolExecutor(max_workers=max_worker) as executor:
            future_to_doc = {executor.submit(fetch_document, api_key, doc_id): doc_id for doc_id in new_document_ids}
            for future in as_completed(future_to_doc):
                try:
                    result, record_time = future.result()
                    total_record_time += record_time
                    if result:
                        documents.append(result)
                        dataset_documents.append(result)
                    if len(documents) >= N:
                        break
                except Exception as e:
                    print(f"An error occurred: {e}")
        
        if len(dataset_documents) >= documents_per_dataset:
            break
        
        time.sleep(0.05) 
        
        if len(documents) >= N:
            break

    if len(documents) >= N:
        break

print(f"Total time spent on search API calls: {total_search_time} seconds")
print(f"Total time spent on record API calls: {total_record_time/max_worker} seconds")


Number of documents to fetch: 1000000
Number of documents per dataset: 1392
Number of datasets: 718
Number of requests per dataset: 100
number of documents from this dataset: 0
Request #1 for dataset 2058621_Ag_EU_LoCloud_NRA


/tmp/ipykernel_3700205/4101135485.py:67: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dataset_name = row[0]


total number of documents: 100
number of documents from this dataset: 100
Request #2 for dataset 2058621_Ag_EU_LoCloud_NRA
total number of documents: 200
number of documents from this dataset: 200
Request #3 for dataset 2058621_Ag_EU_LoCloud_NRA
total number of documents: 300
number of documents from this dataset: 300
Request #4 for dataset 2058621_Ag_EU_LoCloud_NRA
total number of documents: 400
number of documents from this dataset: 400
Request #5 for dataset 2058621_Ag_EU_LoCloud_NRA
total number of documents: 500
number of documents from this dataset: 500
Request #6 for dataset 2058621_Ag_EU_LoCloud_NRA
total number of documents: 600
number of documents from this dataset: 600
Request #7 for dataset 2058621_Ag_EU_LoCloud_NRA
total number of documents: 700
number of documents from this dataset: 700
Request #8 for dataset 2058621_Ag_EU_LoCloud_NRA
total number of documents: 800
number of documents from this dataset: 800
Request #9 for dataset 2058621_Ag_EU_LoCloud_NRA
total number of 

In [6]:
# Check the number of documents collected
print(f'Total documents collected: {len(documents)}')

Total documents collected: 996796


In [7]:
def save_documents(dataset_name, documents):
    if not os.path.exists(dataset_name):
        os.makedirs(dataset_name)

    for i, doc in enumerate(documents):
        with open(f'{dataset_name}/document_{i}.rdf', 'w') as f:
            f.write(doc)

save_documents("set_1", documents)

In [1]:
!ls

 2021672				        output_solr2.xml
 data_collection_2.ipynb		        output_solr33.xml
 data_collection2.py			        output_solr34.xml
 data_collection.ipynb			        output_solr3.xml
 data_collection.py			        output_solr.xml
 data_collection_script.ipynb		        parser_1.py
 datapipeline.ipynb			        parser_2.py
'europeana_datasets - europeana_datasets.csv'   parser2test.py
'Europeana documents'			        __pycache__
 hunnid					        set_1


In [3]:
import os
files_list = os.listdir('set_1')

In [8]:
import os

def get_folder_size_in_gb(folder_path):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(folder_path):
        for filename in filenames:
            file_path = os.path.join(dirpath, filename)
            total_size += os.path.getsize(file_path)
    
    # Convert the total size from bytes to gigabytes
    total_size_gb = total_size / (1024 ** 3)
    return total_size_gb

# Specify the folder path
folder_path = 'set_1'

# Get the folder size in GB
folder_size_gb = get_folder_size_in_gb(folder_path)

# Print the folder size
print(f"Size of the folder '{folder_path}' is: {folder_size_gb:.2f} GB")


Size of the folder 'set_1' is: 18.93 GB


In [7]:
files_list[0]

'document_358560.rdf'

In [12]:
import os

# Define the folder path
folder_path = 'set_1'

# List all RDF files in the folder
rdf_files = [f for f in os.listdir(folder_path) if f.endswith('.rdf')]

# Check if there are RDF files in the folder
if rdf_files:
    # Get the first RDF file
    first_rdf_file = rdf_files[0]
    
    # Construct the full path to the first RDF file
    file_path = os.path.join(folder_path, first_rdf_file)
    
    # Open and read the contents of the RDF file
    with open(file_path, 'r', encoding='utf-8') as file:
        rdf_content = file.read()
    
    # Display the entire content of the RDF file
    print(f"Contents of {first_rdf_file}:\n")
    display(rdf_content)
else:
    print("No RDF files found in the folder.")


Contents of document_358560.rdf:



'<?xml version="1.0" encoding="UTF-8" standalone="yes"?><rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#" xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:dcterms="http://purl.org/dc/terms/" xmlns:edm="http://www.europeana.eu/schemas/edm/" xmlns:owl="http://www.w3.org/2002/07/owl#" xmlns:wgs84_pos="http://www.w3.org/2003/01/geo/wgs84_pos#" xmlns:skos="http://www.w3.org/2004/02/skos/core#" xmlns:rdaGr2="http://rdvocab.info/ElementsGr2/" xmlns:foaf="http://xmlns.com/foaf/0.1/" xmlns:ebucore="http://www.ebu.ch/metadata/ontologies/ebucore/ebucore#" xmlns:doap="http://usefulinc.com/ns/doap#" xmlns:odrl="http://www.w3.org/ns/odrl/2/" xmlns:cc="http://creativecommons.org/ns#" xmlns:ore="http://www.openarchives.org/ore/terms/" xmlns:svcs="http://rdfs.org/sioc/services#" xmlns:oa="http://www.w3.org/ns/oa#" xmlns:dqv="http://www.w3.org/ns/dqv#"><edm:ProvidedCHO rdf:about="http://data.europeana.eu/item/991/https___catalonica_bnc_cat_catalonicahub_lod_oai_cdm21055_contentdm_oclc_or